# Web Scraping AmbitionBox Companies Data

This notebook scrapes top company details from **AmbitionBox** using `requests`, `BeautifulSoup`, and `pandas`.

> **Note on Updated Tags:** AmbitionBox has updated its HTML structure. The old tags (`company-content-wrapper`, `infoEntity`, `p.rating`) have been updated to modern BEM classes (`companyCardWrapper`, `companyCardWrapper__companyName`, `rating_text`, `companyCardWrapper__interLinking`, `companyCardWrapper__ActionWrapper`). This notebook is fully updated and optimized to handle modern tags and export clean CSV datasets.

In [71]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np
import time
import re

## 1. Setting Headers (Preventing HTTP 403 Forbidden)
Websites often block automated scrapers if standard browser request headers are missing. We supply a realistic `User-Agent` and `Accept` headers.

In [72]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Referer': 'https://www.google.com/'
}

# Test single page request
url = 'https://www.ambitionbox.com/list-of-companies?page=1'
response = requests.get(url, headers=headers)
print(f'Status Code: {response.status_code}')

Status Code: 200


In [73]:
soup = BeautifulSoup(response.text, 'lxml')
# Verify page heading
page_title = soup.find('h1').text.strip() if soup.find('h1') else 'AmbitionBox Companies'
print('Page Title:', page_title)

Page Title: Top Companies in
				 
					INDIA


## 2. Inspecting Modern Container and Tags
- **Company Container:** `<div class="companyCardWrapper">`
- **Company Name:** `<h2 class="companyCardWrapper__companyName">`
- **Rating:** `<div class="rating_text">`
- **Industry & Locations:** `<span class="companyCardWrapper__interLinking">`
- **Actions (Reviews, Salaries, Interviews, Jobs, Benefits):** `<a class="companyCardWrapper__ActionWrapper">`
- **Sentiment Highlights (Highly/Critically Rated For):** `<span class="companyCardWrapper__ratingHeader">` & `<span class="companyCardWrapper__ratingValues">`

In [74]:
# Find all company cards on the page
companies = soup.find_all('div', class_=lambda c: c and 'companyCardWrapper' in c.split())
print(f'Total company cards found on page 1: {len(companies)}')

Total company cards found on page 1: 20


## 3. Extracting Data for a Single Page

In [75]:
name = []
full_name = []
rating = []
reviews = []
salaries = []
interviews = []
jobs = []
benefits = []
industry = []
hq_locations = []
rated_highlight = []
profile_url = []

for card in companies:
    # 1. Company Name
    name_el = card.find('h2', class_=re.compile(r'companyCardWrapper__companyName'))
    name.append(name_el.text.strip() if name_el else np.nan)
    
    # Full Name / Alternate Name
    full_name_meta = card.find('meta', itemprop='alternateName')
    full_name.append(full_name_meta['content'].strip() if full_name_meta and full_name_meta.get('content') else (name_el.text.strip() if name_el else np.nan))
    
    # 2. Rating
    rating_el = card.find('div', class_=re.compile(r'rating_text'))
    rating.append(rating_el.text.strip() if rating_el else np.nan)
    
    # 3. Action Counts (Reviews, Salaries, Interviews, Jobs, Benefits)
    actions = {}
    for action in card.find_all('a', class_=re.compile(r'companyCardWrapper__ActionWrapper')):
        count_el = action.find('span', class_=re.compile(r'companyCardWrapper__ActionCount'))
        title_el = action.find('span', class_=re.compile(r'companyCardWrapper__ActionTitle'))
        if count_el and title_el:
            actions[title_el.text.strip()] = count_el.text.strip()
            
    reviews.append(actions.get('Reviews', np.nan))
    salaries.append(actions.get('Salaries', np.nan))
    interviews.append(actions.get('Interviews', np.nan))
    jobs.append(actions.get('Jobs', np.nan))
    benefits.append(actions.get('Benefits', np.nan))
    
    # 4. Industry & Locations
    interlink_el = card.find('span', class_=re.compile(r'companyCardWrapper__interLinking'))
    parts = [p.strip() for p in interlink_el.text.strip().split('|')] if interlink_el else []
    industry.append(parts[0] if len(parts) > 0 else np.nan)
    hq_locations.append(parts[1] if len(parts) > 1 else np.nan)
    
    # 5. Rated For Highlights
    rated_val_el = card.find('span', class_=re.compile(r'companyCardWrapper__ratingValues'))
    rated_highlight.append(rated_val_el.text.strip() if rated_val_el else np.nan)
    
    # 6. Profile URL
    url_meta = card.find('meta', itemprop='url')
    profile_url.append(url_meta['content'].strip() if url_meta and url_meta.get('content') else np.nan)

# Build DataFrame for Page 1
df = pd.DataFrame({
    'Company_Name': name,
    'Full_Name': full_name,
    'Rating': rating,
    'Reviews': reviews,
    'Salaries': salaries,
    'Interviews': interviews,
    'Jobs': jobs,
    'Benefits': benefits,
    'Industry': industry,
    'Headquarters_Locations': hq_locations,
    'Rated_For': rated_highlight,
    'Profile_URL': profile_url
})

df.head()

,Company_Name,Full_Name,Rating,Reviews,Salaries,Interviews,Jobs,Benefits,Industry,Headquarters_Locations,Rated_For,Profile_URL
0,TCS,Tata Consultancy Services,3.2,1.2L,10.4L,11.4k,5.3k,11.1k,IT Services & Consulting,Bengaluru +479 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/tcs-overview
1,Accenture,Accenture overview,3.7,77.3k,7.3L,9.6k,14.9k,7k,IT Services & Consulting,Bengaluru +283 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/accenture...
2,Wipro,Wipro overview,3.6,68k,4.9L,7k,7,5k,IT Services & Consulting,Hyderabad +393 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/wipro-ove...
3,Cognizant,Cognizant overview,3.7,64.2k,6.1L,6.6k,810,5.7k,IT Services & Consulting,Hyderabad +254 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/cognizant...
4,Capgemini,Capgemini overview,3.6,56.4k,5L,5.7k,2.1k,3.9k,IT Services & Consulting,Bengaluru +206 other locations,"Work Life Balance, Job Security",https://www.ambitionbox.com/overview/capgemini...


In [76]:
df.shape

(20, 12)

## 4. High-Performance Multi-Page Web Scraping

### Performance Best Practices Applied:
1. **`requests.Session()`**: Reuses TCP connections for significantly faster HTTP requests.
2. **List of Dicts Accumulation**: Avoids deprecated `DataFrame.append()` which slows down exponentially.
3. **Polite Delay (`time.sleep`)**: Prevents IP rate-limiting and blocks.
4. **Robust Error Handling (`try-except`)**: Handles missing fields gracefully without breaking the loop.
5. **Configurable Page Range**: Set `TOTAL_PAGES` to scrape as many pages as desired (e.g., 5 to 50+).

In [77]:
# Specify the number of pages to scrape (e.g. 5 pages = 100 companies)
START_PAGE = 1
TOTAL_PAGES = 5  # Adjust as needed (e.g. 10, 20, 50)

all_companies = []
session = requests.Session()
session.headers.update(headers)

for page in range(START_PAGE, START_PAGE + TOTAL_PAGES):
    url = f'https://www.ambitionbox.com/list-of-companies?page={page}'
    print(f'Scraping Page {page} of {START_PAGE + TOTAL_PAGES - 1}...', end=' ')
    
    try:
        resp = session.get(url, timeout=12)
        if resp.status_code != 200:
            print(f'Failed (Status {resp.status_code})')
            continue
            
        soup = BeautifulSoup(resp.text, 'lxml')
        cards = soup.find_all('div', class_=lambda c: c and 'companyCardWrapper' in c.split())
        
        for card in cards:
            try:
                name_el = card.find('h2', class_=re.compile(r'companyCardWrapper__companyName'))
                c_name = name_el.text.strip() if name_el else np.nan
                
                full_name_meta = card.find('meta', itemprop='alternateName')
                c_full_name = full_name_meta['content'].strip() if full_name_meta and full_name_meta.get('content') else c_name
                
                rating_el = card.find('div', class_=re.compile(r'rating_text'))
                c_rating = rating_el.text.strip() if rating_el else np.nan
                
                actions = {}
                for action in card.find_all('a', class_=re.compile(r'companyCardWrapper__ActionWrapper')):
                    count_el = action.find('span', class_=re.compile(r'companyCardWrapper__ActionCount'))
                    title_el = action.find('span', class_=re.compile(r'companyCardWrapper__ActionTitle'))
                    if count_el and title_el:
                        actions[title_el.text.strip()] = count_el.text.strip()
                        
                interlink_el = card.find('span', class_=re.compile(r'companyCardWrapper__interLinking'))
                parts = [p.strip() for p in interlink_el.text.strip().split('|')] if interlink_el else []
                c_industry = parts[0] if len(parts) > 0 else np.nan
                c_location = parts[1] if len(parts) > 1 else np.nan
                
                rated_val_el = card.find('span', class_=re.compile(r'companyCardWrapper__ratingValues'))
                c_rated_for = rated_val_el.text.strip() if rated_val_el else np.nan
                
                url_meta = card.find('meta', itemprop='url')
                c_url = url_meta['content'].strip() if url_meta and url_meta.get('content') else np.nan
                
                all_companies.append({
                    'Company_Name': c_name,
                    'Full_Name': c_full_name,
                    'Rating': c_rating,
                    'Reviews': actions.get('Reviews', np.nan),
                    'Salaries': actions.get('Salaries', np.nan),
                    'Interviews': actions.get('Interviews', np.nan),
                    'Jobs': actions.get('Jobs', np.nan),
                    'Benefits': actions.get('Benefits', np.nan),
                    'Industry': c_industry,
                    'Headquarters_Locations': c_location,
                    'Rated_For': c_rated_for,
                    'Profile_URL': c_url
                })
            except Exception as item_err:
                continue
                
        print(f'Done ({len(cards)} companies extracted)')
        time.sleep(1)  # Polite crawling delay
    except Exception as e:
        print(f'Error: {e}')

# Construct the final DataFrame
final = pd.DataFrame(all_companies)
print(f'\nScraping Complete! Total companies gathered: {len(final)}')

Scraping Page 1 of 5... Done (20 companies extracted)
Scraping Page 2 of 5... Done (20 companies extracted)
Scraping Page 3 of 5... Done (20 companies extracted)
Scraping Page 4 of 5... Done (20 companies extracted)
Scraping Page 5 of 5... Done (20 companies extracted)

Scraping Complete! Total companies gathered: 100


## 5. Exploring the Final Dataset

In [78]:
final.head(10)

,Company_Name,Full_Name,Rating,Reviews,Salaries,Interviews,Jobs,Benefits,Industry,Headquarters_Locations,Rated_For,Profile_URL
0,TCS,Tata Consultancy Services,3.2,1.2L,10.4L,11.4k,5.3k,11.1k,IT Services & Consulting,Bengaluru +479 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/tcs-overview
1,Accenture,Accenture overview,3.7,77.3k,7.3L,9.6k,14.9k,7k,IT Services & Consulting,Bengaluru +283 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/accenture...
2,Wipro,Wipro overview,3.6,68k,4.9L,7k,7,5k,IT Services & Consulting,Hyderabad +393 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/wipro-ove...
3,Cognizant,Cognizant overview,3.7,64.2k,6.1L,6.6k,810,5.7k,IT Services & Consulting,Hyderabad +254 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/cognizant...
4,Capgemini,Capgemini overview,3.6,56.4k,5L,5.7k,2.1k,3.9k,IT Services & Consulting,Bengaluru +206 other locations,"Work Life Balance, Job Security",https://www.ambitionbox.com/overview/capgemini...
5,HDFC Bank,HDFC Bank overview,3.8,55.5k,1.5L,3.2k,358,3.5k,Banking,Mumbai +1912 other locations,Job Security,https://www.ambitionbox.com/overview/hdfc-bank...
6,Infosys,Infosys overview,3.5,51.2k,5.4L,8.6k,4.3k,5k,IT Services & Consulting,Bengaluru +259 other locations,Job Security,https://www.ambitionbox.com/overview/infosys-o...
7,HCLTech,HCL Technologies,3.4,48.4k,4L,4.7k,343,4k,IT Services & Consulting,Bengaluru +250 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/hcl-techn...
8,ICICI Bank,ICICI Bank overview,4.0,47.6k,1.6L,3.1k,24,3.8k,Banking,Mumbai +1479 other locations,"Job Security, Promotions, Skill Development",https://www.ambitionbox.com/overview/icici-ban...
9,Tech Mahindra,Tech Mahindra overview,3.3,45.4k,2.8L,4.7k,813,3.5k,IT Services & Consulting,Hyderabad +340 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/tech-mahi...


In [79]:
final.sample(5)

,Company_Name,Full_Name,Rating,Reviews,Salaries,Interviews,Jobs,Benefits,Industry,Headquarters_Locations,Rated_For,Profile_URL
21,Larsen & Toubro Limited,Larsen and Toubro,3.9,25.2k,77.6k,2k,276,2.6k,Engineering & Construction,Mumbai +807 other locations,"Job Security, Skill Development, Salary",https://www.ambitionbox.com/overview/larsen-an...
64,JSW Steel,JSW Steel overview,3.9,8.4k,32.6k,899,147,658,Iron & Steel,Ballari +208 other locations,"Job Security, Skill Development",https://www.ambitionbox.com/overview/jsw-steel...
86,Bajaj Life Insurance,Bajaj Life Insurance overview,3.8,7.1k,18.3k,467,112,512,Insurance,Pune +632 other locations,Salary,https://www.ambitionbox.com/overview/bajaj-lif...
79,Amazon Development Centre India,AMAZON DEVELOPMENT CENTRE INDIA PVT LTD,3.8,7.6k,39.6k,918,22,612,Internet,Hyderabad +148 other locations,"Company Culture, Work Life Balance",https://www.ambitionbox.com/overview/amazon-de...
29,IDFC FIRST Bank,IDFC Bank,4.0,16k,51.4k,1.1k,102,837,Banking,Mumbai +755 other locations,"Salary, Skill Development, Company Culture",https://www.ambitionbox.com/overview/idfc-firs...


In [80]:
final.shape

(100, 12)

In [81]:
final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Company_Name            100 non-null    object
 1   Full_Name               100 non-null    object
 2   Rating                  100 non-null    object
 3   Reviews                 100 non-null    object
 4   Salaries                100 non-null    object
 5   Interviews              100 non-null    object
 6   Jobs                    100 non-null    object
 7   Benefits                100 non-null    object
 8   Industry                100 non-null    object
 9   Headquarters_Locations  100 non-null    object
 10  Rated_For               99 non-null     object
 11  Profile_URL             100 non-null    object
dtypes: object(12)
memory usage: 9.5+ KB


## 6. Exporting the Scraped Data to CSV
Save the final dataset to a `.csv` file format with UTF-8 encoding.

In [82]:
output_filename = 'World_Companies_Data.csv'
final.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f'Successfully exported dataset to {output_filename}!')

Successfully exported dataset to World_Companies_Data.csv!


In [83]:
# Verification by reading the generated CSV back
saved_df = pd.read_csv(output_filename)
print(f'CSV Verification: {saved_df.shape[0]} rows, {saved_df.shape[1]} columns')
saved_df.head(5)

CSV Verification: 100 rows, 12 columns


,Company_Name,Full_Name,Rating,Reviews,Salaries,Interviews,Jobs,Benefits,Industry,Headquarters_Locations,Rated_For,Profile_URL
0,TCS,Tata Consultancy Services,3.2,1.2L,10.4L,11.4k,5.3k,11.1k,IT Services & Consulting,Bengaluru +479 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/tcs-overview
1,Accenture,Accenture overview,3.7,77.3k,7.3L,9.6k,14.9k,7k,IT Services & Consulting,Bengaluru +283 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/accenture...
2,Wipro,Wipro overview,3.6,68k,4.9L,7k,7,5k,IT Services & Consulting,Hyderabad +393 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/wipro-ove...
3,Cognizant,Cognizant overview,3.7,64.2k,6.1L,6.6k,810,5.7k,IT Services & Consulting,Hyderabad +254 other locations,"Promotions, Salary, Work Satisfaction",https://www.ambitionbox.com/overview/cognizant...
4,Capgemini,Capgemini overview,3.6,56.4k,5L,5.7k,2.1k,3.9k,IT Services & Consulting,Bengaluru +206 other locations,"Work Life Balance, Job Security",https://www.ambitionbox.com/overview/capgemini...
